[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/WesamKhallaf73/pytorch-ground-up-notebooks/blob/main/notebooks/L05.ipynb)

# Lecture 05 — Reshaping, Squeezing & Adding Axes

*Same numbers, different grid — and the tiny size-1 axes that make everything else fit.*

**PyTorch from Ground Up · Volume I · Chapters 6–7** — companion notebook.

**How to use it:** run the cells from top to bottom. Before each code cell, *predict* what it will print — then run it and compare with the expected output (click ▸ to open it). Every exercise from the lecture is at the end.

By the end you will be able to:
- Reshape a tensor and state the one constraint every reshape obeys.
- Explain `view` vs `reshape`, and what “contiguous” means.
- Use `-1` to infer a dimension and `flatten(1)` to keep the batch.
- Add and remove size-1 axes with `unsqueeze`, `None` and `squeeze`.
- Apply the three model moves: add a batch dim, shape for broadcasting, drop a trailing dim.
- Explain why all of this is free.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
print('PyTorch', torch.__version__)

## 01 · Data and shape are two different things

A tensor’s **data** is one flat row; its **shape** is just how you look at it. Twelve numbers can be a 3×4, a 4×3, a 2×6 or a flat 12 — same bytes, different grid.

<sub>Source: lecture slides 3–6</sub>

🧪 *Interactive lab in the app:* **One row, many shapes** — try it there to see this idea move.

## 02 · view vs reshape, and contiguity

**Contiguous** means the grid reads in the same order the memory sits. Most tensors are contiguous from birth. `transpose`, `permute` and `expand` break it — they change the strides without moving the data.

<sub>Source: lecture slides 7–13</sub>

🧪 *Interactive lab in the app:* **Transpose swaps strides, not data** — try it there to see this idea move.

**`reshape.py`** — predict the output, then run.

In [ ]:
t = torch.arange(6).reshape(2, 3)
tt = t.T                        # non-contiguous
# tt.view(6) ← RuntimeError!
print(tt.reshape(6))            # works — copies
print(tt.contiguous().view(6))  # also works

<details><summary>▸ Expected output</summary>

```
tensor([0, 3, 1, 4, 2, 5])
tensor([0, 3, 1, 4, 2, 5])
```
</details>

> ⚠️ **Watch out — reshape may copy — don’t count on sharing**  
> Because `reshape` sometimes returns a view and sometimes a copy, you cannot rely on it to share memory.
>
> If you specifically need a view — so an in-place edit propagates back to the original — use `view`; and if it errors, that’s PyTorch telling you the operation isn’t safe as a view.

## 03 · The -1 shortcut and flatten

Pass `-1` for **one** dimension and PyTorch does the arithmetic.

<sub>Source: lecture slides 14–20</sub>

**`infer.py`** — predict the output, then run.

In [ ]:
t = torch.arange(24)
print(t.reshape(4, -1).shape)   # (4, 6)
print(t.reshape(-1, 3).shape)   # (8, 3)
print(t.reshape(2,3,-1).shape)  # (2, 3, 4)

<details><summary>▸ Expected output</summary>

```
torch.Size([4, 6])
torch.Size([8, 3])
torch.Size([2, 3, 4])
```
</details>

**`flatten.py`** — predict the output, then run.

In [ ]:
img = torch.randn(3, 28, 28)  # C, H, W
flat = img.flatten()          # all dims → 1
print(flat.shape)             # (2352,)
hw = img.flatten(1)           # flatten from dim 1
print(hw.shape)               # (3, 784)

batch = torch.randn(32, 3, 28, 28)  # N,C,H,W
print(batch.flatten(1).shape)       # → (32, 2352)

<details><summary>▸ Expected output</summary>

```
torch.Size([2352])
torch.Size([3, 784])
torch.Size([32, 2352])
```
</details>

> ⚠️ **Watch out — flatten() eats the batch dimension**  
> `flatten()` with no argument removes all dimensions, including the batch — call it in a forward pass and you fuse your whole batch into one long vector.
>
> In a model’s forward pass, always use `flatten(1)` so the batch dimension survives.

## 04 · The missing-dimension problem: unsqueeze and squeeze

The most frustrating shape errors are a dimension **missing** — or one too many. A model wants a batch and you hand it one sample; a loss wants a column and gets a row. The fix is nearly always one call.

<sub>Source: lecture slides 22–28</sub>

**`unsqueeze.py`** — predict the output, then run.

In [ ]:
v = torch.tensor([1, 2, 3, 4])  # shape (4,)
row = v.unsqueeze(0)  # (1, 4) — one row
col = v.unsqueeze(1)  # (4, 1) — one column
print(row.shape, col.shape)

row = v[None, :]  # same as unsqueeze(0)
col = v[:, None]  # same as unsqueeze(1)

<details><summary>▸ Expected output</summary>

```
torch.Size([1, 4]) torch.Size([4, 1])
```
</details>

🧪 *Interactive lab in the app:* **Add and remove axes** — try it there to see this idea move.

> ⚠️ **Watch out — squeeze() can eat your batch dimension**  
> Calling `squeeze()` with no argument removes every size-1 dimension — including the batch dimension, if your batch happens to be size 1. In a training loop that can silently break your shapes.
>
> Prefer `squeeze(dim)` to be explicit about exactly which axis you mean.

## 05 · Three places these show up



<sub>Source: lecture slides 29–32</sub>

_Stand-in: the next listing uses names from your own project (a model, a data loader…). This cell defines small ones so it runs here._

In [ ]:
model = nn.Sequential(nn.Flatten(), nn.LazyLinear(1))
x = torch.randn(32, 3, 28, 28)
B, S, H, D = 2, 10, 4, 64


**`three_moves.py`** — predict the output, then run.

In [ ]:
# 1 · Adding a batch dimension
sample = torch.randn(3, 28, 28)  # one image
batch = sample.unsqueeze(0)      # (1, 3, 28, 28)
output = model(batch)            # now it works

# 2 · Preparing for broadcasting
mean = torch.tensor([0.485, 0.456, 0.406])  # (3,)
mean = mean[:, None, None]                  # (3,1,1)
img = torch.randn(3, 224, 224)
img = img - mean   # broadcasts (3,1,1) to (3,224,224)

# 3 · Removing an output dimension
preds = model(x)             # (32, 1)
preds = preds.squeeze(-1)    # (32,)

## 06 · Why it’s all free

`squeeze` and `unsqueeze` are **free**: same memory, new strides. Nothing is allocated, nothing copied — call them as often as you like, even in hot loops.

<sub>Source: lecture slides 33–36</sub>

**`stride_check.py`** — predict the output, then run.

In [ ]:
v = torch.arange(4)
u = v.unsqueeze(0)
print(u.data_ptr() == v.data_ptr())  # True
print(u.stride())                    # (4, 1)

<details><summary>▸ Expected output</summary>

```
True
(4, 1)
```
</details>

## The rule — worth remembering

**Reshaping**

1. Reshaping reinterprets the flat memory under a new shape — same data, new lens.
2. The product of the new shape must equal `numel()`.
3. `view` requires contiguity and always returns a view. `reshape` always works — view if it can, copy if it must.
4. Most tensors are contiguous. `transpose`, `permute`, and `expand` break contiguity.
5. Use `-1` to let PyTorch infer one dimension.
6. `flatten(1)` is the standard pattern before a linear layer — keep the batch dim, collapse the rest.

**Squeeze & unsqueeze**

1. `unsqueeze(dim)` inserts a size-1 axis. `v[None, :]` does the same thing inline.
2. `squeeze(dim)` removes a size-1 axis. Without `dim`, it removes all of them — which can be dangerous.
3. Both are free views — no data is copied.
4. Use `unsqueeze(0)` to add a batch dimension for a single sample.
5. Use `unsqueeze` or `[:, None, None]` to align shapes for broadcasting.
6. Use `squeeze(-1)` to remove a trailing singleton from model outputs.

## Pause and try

Work each exercise in the empty cell below it. Open the worked answer only after you have tried.

### Exercise 01 · Reshape

Take `arange(24)` and list every valid 2-D shape — how many are there? Reshape `arange(60)` into `(3, -1, 5)` — what’s the inferred middle dimension?

In [ ]:
# your code here


<details><summary>▸ Worked answer</summary>

24 has eight divisors, so there are 8 valid 2-D shapes: (1,24), (2,12), (3,8), (4,6), (6,4), (8,3), (12,2), (24,1). For 60 elements, `(3, -1, 5)` infers 60 ÷ 15 = **4**.
</details>

### Exercise 02 · Contiguity

Make a (2, 3) tensor, transpose it, check `.is_contiguous()`; call `.contiguous()` and check again; print strides before and after. Try `.view(6)` on the transpose — what happens? Fix it with `.reshape(6)`.

In [ ]:
# your code here


<details><summary>▸ Worked answer</summary>

After `.T` the tensor is non-contiguous with strides (1, 3); `.view(6)` raises a RuntimeError. `.contiguous()` copies the data into grid order (strides become (2, 1)) and `.is_contiguous()` is True. `.reshape(6)` works directly because it copies when it must.
</details>

### Exercise 03 · Flatten

Flatten a (16, 3, 32, 32) batch from dimension 1 — what shape do you get?

In [ ]:
# your code here


<details><summary>▸ Worked answer</summary>

`(16, 3072)` — the batch of 16 survives, and 3 × 32 × 32 = 3072.
</details>

### Exercise 04 · Axes

Turn a (5,) vector into (1, 5), (5, 1), and (1, 5, 1) with `unsqueeze`, then again with `None` indexing. Shape the ImageNet std `[0.229, 0.224, 0.225]` so you can divide a (3, 224, 224) image by it.

In [ ]:
# your code here


<details><summary>▸ Worked answer</summary>

`v.unsqueeze(0)`, `v.unsqueeze(1)`, `v.unsqueeze(0).unsqueeze(-1)`; or `v[None, :]`, `v[:, None]`, `v[None, :, None]`. The std needs shape (3, 1, 1): `std[:, None, None]`.
</details>

### Interview-style

What’s the difference between `.view()` and `.reshape()`, and when does it matter? (Three sentences, mention contiguity and memory.)

_Write your answer here before opening the model answer._

<details><summary>▸ Model answer</summary>

`view` always returns a view that shares memory with the original, and it requires the tensor to be contiguous — otherwise it raises an error. `reshape` returns a view when it can and silently copies when it must, so it never fails on contiguity. The difference only shows up after operations like `transpose` or `permute`, and it matters when you need guaranteed memory sharing (use `view`) versus guaranteed success (use `reshape`).
</details>

### Interview-style

Your model expects (N, C, H, W) but you have a single image (C, H, W) — how do you pass it through? (One sentence, name the operation and the resulting shape.)

_Write your answer here before opening the model answer._

<details><summary>▸ Model answer</summary>

Call `img.unsqueeze(0)` to add a batch dimension of size 1, giving shape (1, C, H, W).
</details>